# 🔬 M5Stamp ESP32-S3: Reinforcement Learning Edge AI Fire & Gas Monitoring
### Standalone MDP Bellman Training, Weight Optimization & Policy Evaluation Notebook
**Target Microcontroller:** M5Stamp ESP32-S3 (Xtensa Dual-Core 240MHz, 512KB SRAM)  
**Sensors:** Bosch BME680 (VOC Gas, Temperature), INA219 (Battery Fuel Gauge), 1S LiFePO4 Cell  
**RL Formulation:** Markov Decision Process (MDP) $\langle \mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma 
angle$ with 54 Discrete States & 4 Duty-Cycling Actions

---
This notebook is **100% self-contained** and operates independently of Wokwi, cloud backends, and frontends. It demonstrates:
1. **Closed-Loop Gymnasium Environment ()**: Simulating micro-hotplate pulse heating (1.15 mJ), deep sleep (15 µA), Wi-Fi bursts (576 mJ), and battery dynamics.
2. **1,200 MDP Experience Transitions**: Gathering $ across Clean Air, Heat Waves, Toxic Gas Leaks, Smoldering Fires, and Flaming Fires.
3. **Bellman Temporal Difference (TD) Learning**: Updating Q-table weights via $\Delta Q = Alpha [r + \gamma \max Q(s') - Q(s, a)]$.
4. **Loss & Reward Convergence**: Showing a 98.9% reduction in Bellman TD loss and return climbing from -2,280 to +315.
5. **4x4 Policy Decision Confusion Matrix**: Proving **100% Hazard Recall (Zero False Negatives)** on emergency alerts.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

from env import AdaptiveAirQualityEnv
from standalone_rl_demo import QLearningAgent, collect_mdp_transitions, evaluate_rl_agent, ACTION_NAMES, SCENARIO_NAMES

np.random.seed(42)
print("✓ RL Environment, Agent, and Analysis tools loaded!")

✓ RL Environment, Agent, and Analysis tools loaded!


## 1. Collect 1,200 MDP Experience Transitions
We simulate edge agent interaction with  across 5 hazard modes.

In [2]:
df_transitions = collect_mdp_transitions(1200)
print(f"Collected {len(df_transitions)} MDP transitions.")
print("--- Hazard Regime Breakdown ---")
print(df_transitions["scenario_name"].value_counts())
df_transitions.head(5)

Collected 1200 MDP transitions.
--- Hazard Regime Breakdown ---
scenario_name
Clean Ambient Air               430
Heat Wave / Elevated Ambient    332
Flaming / Flashover Fire        209
Toxic VOC / Gas Leak            195
Smoldering / Incipient Fire      34
Name: count, dtype: int64


,step_id,scenario_id,scenario_name,state_idx,temp_c,gas_r_kohm,gas_deriv,battery_soc_pct,action,action_name,optimal_action,optimal_action_name,reward,next_state_idx,next_temp_c,next_gas_r_kohm,done
0,1,3,Smoldering / Incipient Fire,2,23.08,49.70,0.000,78.6,0,DEEP_SLEEP_LONG,1,NORMAL_CYCLE,-0.1502,2,23.16,49.85,False
1,2,3,Smoldering / Incipient Fire,2,21.20,49.70,0.000,78.6,3,EMERGENCY_ALERT,1,NORMAL_CYCLE,-82.8922,2,23.19,49.78,False
2,3,3,Smoldering / Incipient Fire,2,22.76,49.59,-0.053,78.6,1,NORMAL_CYCLE,1,NORMAL_CYCLE,-0.0262,2,23.22,49.79,False
3,4,3,Smoldering / Incipient Fire,2,23.11,49.61,0.000,78.6,3,EMERGENCY_ALERT,1,NORMAL_CYCLE,-82.8922,2,23.26,49.87,False
4,5,3,Smoldering / Incipient Fire,2,23.04,50.21,0.299,78.6,0,DEEP_SLEEP_LONG,1,NORMAL_CYCLE,-0.1502,2,23.28,49.96,False


## 2. Train RL Agent via Bellman Equation (Weight Updates)
We execute Temporal Difference learning: (s, a) \leftarrow Q(s, a) + Alpha [r + \gamma \max_{a'} Q(s', a') - Q(s, a)]$.

In [3]:
agent = QLearningAgent(num_states=54, num_actions=4, alpha=0.15, gamma=0.95)
history, ep_rewards = agent.train_episodes(num_episodes=1500)
print(f"✓ Training complete! Initial TD Loss: {history['td_loss'][0]:.2f} -> Final TD Loss: {history['td_loss'][-1]:.2f}")

Starting Q-Learning training for 1500 simulated MDP episodes...
Episode    1/1500 | TD Loss (MSE):   57832.64 | Avg Return: -2280.6 | Policy Acc:  92.6% | ||ΔQ||:  6.9370
Episode  150/1500 | TD Loss (MSE):   13305.78 | Avg Return: -1260.9 | Policy Acc:  87.0% | ||ΔQ||:  7.1727


Episode  300/1500 | TD Loss (MSE):    7279.27 | Avg Return: -1242.1 | Policy Acc:  87.0% | ||ΔQ||:  3.2121
Episode  450/1500 | TD Loss (MSE):    7687.85 | Avg Return:  -880.9 | Policy Acc:  90.7% | ||ΔQ||:  0.9324


Episode  600/1500 | TD Loss (MSE):    4341.06 | Avg Return:  -491.1 | Policy Acc:  88.9% | ||ΔQ||:  2.5431
Episode  750/1500 | TD Loss (MSE):     293.47 | Avg Return:  -252.5 | Policy Acc:  92.6% | ||ΔQ||:  0.6599


Episode  900/1500 | TD Loss (MSE):    3656.84 | Avg Return:     4.0 | Policy Acc:  87.0% | ||ΔQ||:  1.4964
Episode 1050/1500 | TD Loss (MSE):       8.33 | Avg Return:   112.7 | Policy Acc:  85.2% | ||ΔQ||:  0.1178


Episode 1200/1500 | TD Loss (MSE):     736.34 | Avg Return:   191.2 | Policy Acc:  87.0% | ||ΔQ||:  0.6633
Episode 1350/1500 | TD Loss (MSE):      47.82 | Avg Return:   152.4 | Policy Acc:  87.0% | ||ΔQ||:  0.2167


Episode 1500/1500 | TD Loss (MSE):     661.20 | Avg Return:   315.8 | Policy Acc:  90.7% | ||ΔQ||:  0.4095
✓ Training complete! Initial TD Loss: 57832.64 -> Final TD Loss: 661.20


## 3. RL Policy Evaluation & 4x4 Confusion Matrix
We evaluate greedy policy $\pi^*(s) = rg\max_a Q(s, a)$ across all 1,200 environmental benchmark states.

In [4]:
acc, cm, cr, y_true, y_pred = evaluate_rl_agent(agent, df_transitions)
print(f"Overall Policy Accuracy: {acc * 100:.2f}%")
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=ACTION_NAMES, digits=3, zero_division=0))

# Zero False Negatives Life Safety Verification
hazard_true = np.sum(y_true == 3)
hazard_pred = np.sum((y_pred == 3) & (y_true == 3))
print(f"Hazard Recall: {hazard_pred}/{hazard_true} ({(hazard_pred/hazard_true)*100:.1f}%) -> ZERO FALSE NEGATIVES!")

Overall Policy Accuracy: 80.67%
Classification Report:
                 precision    recall  f1-score   support

DEEP_SLEEP_LONG      0.000     0.000     0.000        57
   NORMAL_CYCLE      0.784     1.000     0.879       840
  ELEVATED_SCAN      1.000     0.342     0.510       266
EMERGENCY_ALERT      1.000     1.000     1.000        37

       accuracy                          0.807      1200
      macro avg      0.696     0.586     0.597      1200
   weighted avg      0.801     0.807     0.759      1200

Hazard Recall: 37/37 (100.0%) -> ZERO FALSE NEGATIVES!


## 4. Visualizing Bellman TD Loss, Return & Confusion Matrix

In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# Loss Curve
ax1.plot(history["epoch"], history["td_loss"], color="#2563eb", lw=2.2, marker="o")
ax1.set_title("A. Bellman TD Loss Reduction (MSE)", fontweight="bold")
ax1.set_xlabel("Episode")
ax1.set_ylabel("TD Loss")
ax1.grid(True, linestyle=":")

# Confusion Matrix
cm_norm = cm.astype("float") / np.maximum(1, cm.sum(axis=1)[:, np.newaxis]) * 100
sns.heatmap(cm_norm, annot=True, fmt=".1f", cmap="Blues", ax=ax2,
            xticklabels=[f"A{i}" for i in range(4)], yticklabels=[f"A{i}" for i in range(4)])
ax2.set_title("B. RL Policy Decision Confusion Matrix (%)", fontweight="bold")
ax2.set_xlabel("RL Chosen Action")
ax2.set_ylabel("Optimal Target Action")
plt.tight_layout()
plt.show()

/tmp/ipykernel_62022/4274224097.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
